# Fine-tuning Microsoft Phi-2 with QLoRA on Synthetic Data

This notebook demonstrates how to:
1. Install required dependencies
2. Generate a small **synthetic instruction dataset**
3. Download the **`microsoft/phi-2`** model in **4-bit (NF4)** with `bitsandbytes`
4. Apply **QLoRA** adapters via PEFT
5. Apply additional optimizations: gradient checkpointing, paged AdamW 8-bit, mixed precision (bf16/fp16), and Flash Attention (when available)
6. Train with the TRL `SFTTrainer`
7. Run inference with the merged adapter

> **Hardware:** A CUDA GPU with ≥ 12 GB VRAM is recommended (e.g., T4, A10, A100). On CPU, training will be impractically slow.

## 1. Install dependencies

In [ ]:
%pip install -q -U \
    "transformers>=4.40.0" \
    "accelerate>=0.30.0" \
    "peft>=0.11.0" \
    "trl>=0.9.0" \
    "datasets>=2.19.0" \
    "bitsandbytes>=0.43.0" \
    "einops" \
    "sentencepiece"

## 2. Imports and environment check

In [ ]:
import os, json, random, torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, PeftModel
from trl import SFTTrainer

print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
    print('BF16 supported :', torch.cuda.is_bf16_supported())

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

## 3. Create a synthetic instruction dataset

We'll programmatically build a small instruction-following dataset consisting of simple math, text transformation, and Q&A tasks. In practice, you'd generate a much larger and more diverse set (e.g., via a teacher LLM).

In [ ]:
def make_synthetic_examples(n_per_kind: int = 200):
    examples = []

    # 1) Addition
    for _ in range(n_per_kind):
        a, b = random.randint(0, 999), random.randint(0, 999)
        examples.append({
            'instruction': f'What is {a} + {b}?',
            'input': '',
            'output': f'{a + b}',
        })

    # 2) Multiplication (small)
    for _ in range(n_per_kind):
        a, b = random.randint(0, 25), random.randint(0, 25)
        examples.append({
            'instruction': f'Compute {a} * {b}.',
            'input': '',
            'output': f'{a * b}',
        })

    # 3) Reverse a word
    words = ['banana', 'transformer', 'phi', 'microsoft', 'qlora', 'tokenizer',
             'gradient', 'attention', 'embedding', 'notebook', 'fine', 'tuning']
    for _ in range(n_per_kind):
        w = random.choice(words)
        examples.append({
            'instruction': 'Reverse the following word.',
            'input': w,
            'output': w[::-1],
        })

    # 4) Uppercase
    sentences = ['hello world', 'i love llms', 'phi-2 is small but mighty',
                 'qlora saves memory', 'fine tuning is fun']
    for _ in range(n_per_kind):
        s = random.choice(sentences)
        examples.append({
            'instruction': 'Convert the text to uppercase.',
            'input': s,
            'output': s.upper(),
        })

    # 5) Simple factual Q&A
    qa = [
        ('What is the capital of France?', 'Paris'),
        ('What color do you get by mixing red and blue?', 'Purple'),
        ('How many continents are there?', 'Seven'),
        ('What is H2O commonly known as?', 'Water'),
        ('Who wrote Romeo and Juliet?', 'William Shakespeare'),
    ]
    for _ in range(n_per_kind):
        q, a = random.choice(qa)
        examples.append({'instruction': q, 'input': '', 'output': a})

    random.shuffle(examples)
    return examples

raw = make_synthetic_examples(200)
print('Total examples:', len(raw))
print('Sample:', json.dumps(raw[0], indent=2))

In [ ]:
PROMPT_TEMPLATE = (
    'Instruct: {instruction}\n'
    '{maybe_input}'
    'Output: {output}'
)

def format_example(ex):
    maybe_input = f'Input: {ex["input"]}\n' if ex['input'] else ''
    text = PROMPT_TEMPLATE.format(
        instruction=ex['instruction'],
        maybe_input=maybe_input,
        output=ex['output'],
    )
    return {'text': text}

ds = Dataset.from_list(raw).map(format_example, remove_columns=['instruction', 'input', 'output'])
ds = ds.train_test_split(test_size=0.05, seed=SEED)
print(ds)
print('---\n', ds['train'][0]['text'])

## 4. Load Phi-2 in 4-bit (QLoRA base)

We use `BitsAndBytesConfig` with **NF4** quantization, double quantization, and a bf16 compute dtype (fallback fp16). This is the QLoRA recipe that drastically reduces VRAM.

In [ ]:
MODEL_ID = 'microsoft/phi-2'

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Try to use Flash Attention 2 for speed/memory; fall back to SDPA otherwise.
attn_impl = 'flash_attention_2'
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        attn_implementation=attn_impl,
    )
except (ImportError, ValueError) as e:
    print(f'Flash Attention 2 unavailable ({e}); falling back to SDPA.')
    attn_impl = 'sdpa'
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        attn_implementation=attn_impl,
    )

model.config.use_cache = False  # required for gradient checkpointing
model.config.pretraining_tp = 1
print('Loaded with attention =', attn_impl)

## 5. Prepare the model for k-bit training and attach LoRA adapters

Phi-2 attention modules are named `q_proj`, `k_proj`, `v_proj`, `dense`; MLP modules are `fc1`, `fc2`. Targeting all of them gives a strong adapter.

In [ ]:
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'dense', 'fc1', 'fc2'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Training arguments and SFTTrainer

Optimizations enabled:
- **4-bit NF4 base** + LoRA adapters (QLoRA)
- **`paged_adamw_8bit`** optimizer (memory-efficient, paged)
- **Gradient checkpointing**
- **bf16 / fp16** mixed precision
- **Gradient accumulation** for larger effective batch size
- **Cosine LR schedule** with warmup

In [ ]:
OUTPUT_DIR = 'phi2-qlora-synthetic'
use_bf16 = compute_dtype == torch.bfloat16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.0,
    max_grad_norm=0.3,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to='none',
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=ds['train'],
    eval_dataset=ds['test'],
    dataset_text_field='text',
    max_seq_length=512,
    packing=True,  # pack short samples to use the context window efficiently
)

## 7. Train

In [ ]:
trainer.train()
trainer.save_model(OUTPUT_DIR)  # saves the LoRA adapter
tokenizer.save_pretrained(OUTPUT_DIR)
print('Adapter saved to', OUTPUT_DIR)

## 8. Inference with the fine-tuned adapter

Reload the 4-bit base model and attach the saved LoRA adapter.

In [ ]:
del model, trainer
torch.cuda.empty_cache() if torch.cuda.is_available() else None

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation=attn_impl,
)
ft_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
ft_model.eval()

def generate(instruction: str, user_input: str = '', max_new_tokens: int = 64):
    maybe_input = f'Input: {user_input}\n' if user_input else ''
    prompt = f'Instruct: {instruction}\n{maybe_input}Output:'
    inputs = tokenizer(prompt, return_tensors='pt').to(ft_model.device)
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return text.strip()

print(generate('What is 17 + 25?'))
print(generate('Reverse the following word.', 'transformer'))
print(generate('Convert the text to uppercase.', 'phi-2 is small but mighty'))
print(generate('What is the capital of France?'))

## 9. (Optional) Merge LoRA into the base weights

Merging requires loading the base model in **fp16/bf16** (not 4-bit). This produces a standalone model you can save and serve without PEFT.

In [ ]:
# Uncomment to merge & save full-precision weights (needs more VRAM/RAM):
# from peft import PeftModel
# base_fp = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, torch_dtype=compute_dtype, device_map='auto', trust_remote_code=True,
# )
# merged = PeftModel.from_pretrained(base_fp, OUTPUT_DIR).merge_and_unload()
# merged.save_pretrained(OUTPUT_DIR + '-merged')
# tokenizer.save_pretrained(OUTPUT_DIR + '-merged')
# print('Merged model saved.')